In [ ]:
# =========================================================
# BEST-OF MERGE (FAST SCORE) + "NO-OVERLAP PICK" (SAFE)
# - 先用纯 numpy 快速算每组 score
# - 每个 n=1..200：把所有 submission 的该组按 score 排序
# - 从最小开始做 overlap 检查，选第一个“无 overlap”的组
# - 输出最终可提交文件：/kaggle/working/submission.csv
# =========================================================

import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from shapely.geometry import Polygon
from shapely.strtree import STRtree

# -------------------------
# 0) INPUTS
# -------------------------
# 你已有 banning/复制，这里保留你的文件列表即可
submission_files = [
    '/kaggle/input/intergration-of-existing-result-current-best/submission.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission1.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission1.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission2.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission3.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission4.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission5.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission6.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission7.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission8.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission9.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission10.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission11.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission12.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission13.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission14.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission15.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission16.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission17.csv',
    '/kaggle/input/intergration-of-existing-result-current-best/submission18.csv',
    '/kaggle/input/notebooke71f06b7cd/submission.csv',
    '/kaggle/input/santa-submission/submission.csv'
]

OUTPUT_FILE = "/kaggle/working/submission.csv"
MAX_N = 200

# -------------------------
# 1) Tree template (float)
# -------------------------
TREE_PTS = np.array([
    (0,0.8),(0.125,0.5),(0.0625,0.5),(0.2,0.25),(0.1,0.25),(0.35,0),
    (0.075,0),(0.075,-0.2),(-0.075,-0.2),(-0.075,0),(-0.35,0),
    (-0.1,0.25),(-0.2,0.25),(-0.0625,0.5),(-0.125,0.5)
], dtype=np.float64)
TX = TREE_PTS[:, 0]
TY = TREE_PTS[:, 1]

def _strip_s(v) -> str:
    s = str(v).strip()
    if len(s) and (s[0] == "s" or s[0] == "S"):
        return s[1:].strip()
    return s

def _load_csv_flexible(fp: str) -> pd.DataFrame:
    """
    兼容一些 results.csv / 变体：
    - 标准：id,x,y,deg
    - 大小写变体：ID / Deg 等
    """
    df = pd.read_csv(fp)

    # normalize column names
    cols = {c.lower(): c for c in df.columns}
    need = ["id", "x", "y", "deg"]

    if all(k in cols for k in need):
        df = df[[cols["id"], cols["x"], cols["y"], cols["deg"]]].copy()
        df.columns = ["id", "x", "y", "deg"]
        return df

    # 有些可能叫 angle / theta
    alt_deg = None
    for k in ["deg", "angle", "theta", "rotation"]:
        if k in cols:
            alt_deg = cols[k]
            break

    if ("id" in cols) and ("x" in cols) and ("y" in cols) and (alt_deg is not None):
        df = df[[cols["id"], cols["x"], cols["y"], alt_deg]].copy()
        df.columns = ["id", "x", "y", "deg"]
        return df

    raise ValueError(f"Unknown submission format columns={list(df.columns)} for file={fp}")

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    -> columns: id(str), group_n(int), item_id(int),
               x_str,y_str,deg_str, x(float),y(float),deg(float)
    """
    d = df.copy()

    # force to string first
    for c in ["id", "x", "y", "deg"]:
        d[c] = d[c].astype(str).str.strip()

    # ✅ FIX: correct split usage (do NOT unpack DataFrame)
    sp = d["id"].str.split("_", n=1, expand=True)
    if sp.shape[1] < 2:
        raise ValueError("id format must be 'NNN_i' (contains '_'). Example: 001_0")

    d["group_n"] = sp[0].astype(int)
    d["item_id"]  = sp[1].astype(int)

    d["x_str"] = d["x"].map(_strip_s)
    d["y_str"] = d["y"].map(_strip_s)
    d["deg_str"] = d["deg"].map(_strip_s)

    d["x"] = d["x_str"].astype(np.float64)
    d["y"] = d["y_str"].astype(np.float64)
    d["deg"] = d["deg_str"].astype(np.float64)

    d = d.sort_values(["group_n", "item_id"]).reset_index(drop=True)
    return d

# -------------------------
# 2) FAST score (no shapely)
# score = side^2 / n
# -------------------------
def score_group_fast(x: np.ndarray, y: np.ndarray, deg: np.ndarray, n: int) -> float:
    rad = np.deg2rad(deg)
    c = np.cos(rad)[:, None]
    s = np.sin(rad)[:, None]
    px = TX[None, :]
    py = TY[None, :]
    X = px * c - py * s + x[:, None]
    Y = px * s + py * c + y[:, None]
    mnx = float(X.min()); mxx = float(X.max())
    mny = float(Y.min()); mxy = float(Y.max())
    side = max(mxx - mnx, mxy - mny)
    return (side * side) / float(n)

def score_submission_fast(d: pd.DataFrame, max_n: int = 200) -> float:
    total = 0.0
    for n in range(1, max_n + 1):
        g = d[d["group_n"] == n]
        if len(g) != n:
            raise ValueError(f"group {n:03d} should have {n} rows, got {len(g)}")
        total += score_group_fast(g["x"].to_numpy(), g["y"].to_numpy(), g["deg"].to_numpy(), n)
    return float(total)

# -------------------------
# 3) Overlap check (shapely) only on candidates
# -------------------------
def _build_polygons(x: np.ndarray, y: np.ndarray, deg: np.ndarray):
    rad = np.deg2rad(deg)
    c = np.cos(rad)
    s = np.sin(rad)
    polys = []
    for i in range(x.shape[0]):
        X = TX * c[i] - TY * s[i] + x[i]
        Y = TX * s[i] + TY * c[i] + y[i]
        polys.append(Polygon(np.stack([X, Y], axis=1)))
    return polys

def _strtree_query_indices(tree, poly, polys):
    res = tree.query(poly)
    if len(res) == 0:
        return []
    first = res[0]
    if isinstance(first, (int, np.integer)):
        return res
    mp = {id(g): i for i, g in enumerate(polys)}
    return [mp.get(id(g), -1) for g in res]

def has_overlap_group(x: np.ndarray, y: np.ndarray, deg: np.ndarray) -> bool:
    if x.shape[0] <= 1:
        return False
    polys = _build_polygons(x, y, deg)
    tree = STRtree(polys)
    for i, p in enumerate(polys):
        cand = _strtree_query_indices(tree, p, polys)
        for j in cand:
            if j < 0 or j == i:
                continue
            if p.intersects(polys[j]) and (not p.touches(polys[j])):
                return True
    return False

# -------------------------
# 4) Load all submissions
# -------------------------
subs = []
print(f"--- Loading {len(submission_files)} files ---")

for fp in submission_files:
    if not os.path.exists(fp):
        print(f"✗ missing: {fp}")
        continue
    try:
        raw = _load_csv_flexible(fp)
        d = normalize_df(raw)
        d = d[(d["group_n"] >= 1) & (d["group_n"] <= MAX_N)].copy()

        # pre-index group -> row indices
        idx_map = d.groupby("group_n").indices

        key = Path(fp).name
        subs.append({"key": key, "path": fp, "df": d, "idx": idx_map})
        print(f"✓ {key:35s} rows={len(d):6d}")
    except Exception as e:
        print(f"✗ failed: {fp} | {e}")

if len(subs) == 0:
    raise RuntimeError("No valid submission files loaded.")

# -------------------------
# 5) Merge per group: best score + no overlap
# -------------------------
selected_rows = []
source_counts = {s["key"]: 0 for s in subs}

print("\n--- Merging per group (pick best score, reject overlaps) ---")
for n in range(1, MAX_N + 1):
    candidates = []

    for s in subs:
        idx = s["idx"].get(n, None)
        if idx is None:
            continue
        g = s["df"].iloc[idx]
        if len(g) != n:
            continue
        sc = score_group_fast(g["x"].to_numpy(), g["y"].to_numpy(), g["deg"].to_numpy(), n)
        candidates.append((sc, s["key"], g))

    if len(candidates) == 0:
        raise RuntimeError(f"No valid candidates found for group n={n:03d}")

    candidates.sort(key=lambda t: t[0])

    picked = None
    for sc, key, g in candidates:
        if not has_overlap_group(g["x"].to_numpy(), g["y"].to_numpy(), g["deg"].to_numpy()):
            picked = (sc, key, g)
            break

    if picked is None:
        # 所有候选都 overlap（极少见）；强行拿最优但会导致提交无效
        sc, key, g = candidates[0]
        picked = (sc, key, g)
        print(f"⚠️ n={n:03d} all candidates overlap; forced pick {key} (will likely be invalid)")

    sc, key, g = picked
    source_counts[key] += 1

    g2 = g.sort_values("item_id").reset_index(drop=True)
    for i in range(n):
        selected_rows.append({
            "id": f"{n:03d}_{i}",
            "x": "s" + str(g2.loc[i, "x_str"]),
            "y": "s" + str(g2.loc[i, "y_str"]),
            "deg": "s" + str(g2.loc[i, "deg_str"]),
        })

    if n % 25 == 0:
        print(f"  ... merged up to n={n:03d}")

merged_df = pd.DataFrame(selected_rows)
merged_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Wrote merged submission -> {OUTPUT_FILE}")

# -------------------------
# 6) Quick score report
# -------------------------
merged_norm = normalize_df(pd.read_csv(OUTPUT_FILE, usecols=["id", "x", "y", "deg"]))
total_fast = score_submission_fast(merged_norm, MAX_N)

print("\n=== FINAL REPORT ===")
print(f"FAST TOTAL SCORE (Σ side^2/n): {total_fast:.12f}")
print("\n--- Source usage ---")
for k, v in sorted(source_counts.items(), key=lambda x: -x[1]):
    if v > 0:
        print(f"{k:35s}  {v:3d} groups ({v/MAX_N*100:5.1f}%)")

print("\n✅ 你最后提交的文件就是这个：")
print("   ", OUTPUT_FILE)